In [7]:
!pip install pandas
!pip install numpy
!pip install duckdb
!pip install holidays


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os
import duckdb
import holidays
import pandas as pd
import time
import requests

Fonte = https://dados.ons.org.br/dataset/geracao-usina-2


Perguntas de Pesquisa: 
 - É possível prever a geração de energia hidrelétrica utilizando dados históricos de geração?
 - Quais períodos do ano apresentam maior e menor geração de energia hidrelétrica?
 - O nível dos reservatórios está relacionado ao aumento do valor cobrado pela energia?

Hipoteses: 
 - O histórico de geração permite prever a geração futura
 -A geração possui padrões sazonais
 -O nível dos reservatórios está relacionado à geração
 -Em epocas de estiagem, o valor da energia aumenta por causa da escassez



In [9]:
import pandas as pd

df = pd.read_csv('data.csv', sep=';')
df.head()

,din_instante,id_subsistema,nom_subsistema,id_estado,nom_estado,cod_modalidadeoperacao,nom_tipousina,nom_tipocombustivel,nom_usina,id_ons,ceg,val_geracao
0,2026-11-01 00:00:00,N,NORTE,AM,AMAZONAS,Pequenas Usinas (MMGD),FOTOVOLTAICA,Fotovoltaica,PQU MMAM MMGD,NaN,-,0.00000
1,2026-11-01 00:00:00,N,NORTE,AM,AMAZONAS,TIPO I,HIDROELÉTRICA,Hidráulica,BALBINA,AMBA,UHE.PH.AM.000190-2.01,205.19444
2,2026-11-01 00:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,UTE JARAQUI,AMJA,UTE.GN.AM.029272-9.01,37.49800
3,2026-11-01 00:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,APARECIDA,AMAA,UTE.GN.AM.027250-7.02,144.30000
4,2026-11-01 00:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,TAMBAQUI,AMTA,UTE.PE.AM.029276-1.01,42.55800


In [20]:
df.describe()

,din_instante
count,10
mean,2020-12-31 23:00:00
min,2020-12-31 23:00:00
25%,2020-12-31 23:00:00
50%,2020-12-31 23:00:00
75%,2020-12-31 23:00:00
max,2020-12-31 23:00:00


Variavel-alvo: val_geracao
Tipo de geração: Hidráulica
Regiao: Sul e Centro-Oeste
Granularidade: Horária
Período 2001 - 2020
Horizonte de previsão: Mensal

Por que escolher a Granularidade Mensal: 
    O conjunto de dados do ONS ja possui os registros, o que permite aproveitar a resolucaos. A geracao hidreletrica pode variar ao longo do tempo, e a granularidade mensal permitiria analisar e identificar isso de uma forma mais detalhada. Isso nos permitiria, alem de tudo, avaliar o impacto financeiro ao longo do tempo.

Horizonte de planejamento para o mes seguinte:
    Permite estimar a geracao do mes seguinte
    Possibilita utilizar os padroes e horarios para estimar e identificar tendencias. 
    Parece um horizonte adequado para um modelo inicial, sem tornar o problema muito complexo.

Nao foi possivel fazer a agregacao dos dados com o pandas porque devido ao grande volume de dados, o processamento apresentou limitacoes de memoria. Diante disso, optou-se pelo uso do DuckDB.
Foi realizado o processo de juncao dos 19 anos em um unico dataframe.

In [21]:
import os
import duckdb

os.makedirs("dados_consolidados", exist_ok=True)
con = duckdb.connect()

In [22]:
for ano_inicio in range(2001, 2021, 2):

    ano_fim = ano_inicio + 1

    arquivos = [
        f"dados_mensais/GERACAO_USINA-2_{ano_inicio}.parquet",
        f"dados_mensais/GERACAO_USINA-2_{ano_fim}.parquet"
    ]

    destino = f"dados_consolidados/geracao_{ano_inicio}_{ano_fim}.parquet"
    con.execute(f"""
        COPY (
        SELECT *
        FROM read_parquet({arquivos})
        )
        TO '{destino}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [23]:
con.execute("""
    COPY (
    SELECT *
    FROM read_parquet('dados_consolidados/*.parquet')
    )
    TO 'geracao2001_2020.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
import duckdb
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('geracao2001_2020.parquet')
ORDER BY din_instante DESC
LIMIT 10
""").df()

df

,din_instante,id_subsistema,nom_subsistema,id_estado,nom_estado,cod_modalidadeoperacao,nom_tipousina,nom_tipocombustivel,nom_usina,id_ons,ceg,val_geracao
0,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,HIDROELÉTRICA,Hidráulica,Balbina,AMBA,UHE.PH.AM.000190-2.01,102.63700000
1,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO III,TÉRMICA,Óleo Combustível,Electron,AMELC,UTE.PE.AM.027120-9.01,0E-8
2,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO III,TÉRMICA,Óleo Diesel,Flores 3,AMFL3,UTE.PE.AM.029192-7.01,0E-8
3,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,Mauá Bloco 4,AMMU4,UTE.PE.AM.002952-1.01,0E-8
4,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,Cristiano Rocha,AMCR,UTE.PE.AM.029499-3.01,65.84700000
5,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,Manauara,AMMA,UTE.GN.AM.029432-2.01,66.96900000
6,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,Tambaqui,AMTA,UTE.PE.AM.029276-1.01,61.71600000
7,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Óleo Diesel,Iranduba,AMIRD,UTE.PE.AM.030274-0.01,0E-8
8,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO III,TÉRMICA,Óleo Diesel,Flores 2,AMTFL2,UTE.PE.AM.029192-7.01,0E-8
9,2020-12-31 23:00:00,N,NORTE,AM,AMAZONAS,TIPO I,TÉRMICA,Gás,Mauá Bloco 3,AMMU3,UTE.PE.AM.002952-1.01,0E-8


Como vamos fazer uma analise Mensal, a seguir, se cria uma coluna com o ano e uma com o mes. 

In [25]:
con.execute("""
    COPY (
    SELECT *,
    EXTRACT(YEAR FROM din_instante) AS Ano
    FROM read_parquet('geracao2001_2020.parquet')
    )
    TO 'geracao_modificada.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [26]:
con.execute("""
    COPY(
    SELECT *,
    EXTRACT (MONTH FROM din_instante) AS MES
    FROM read_parquet('geracao_modificada.parquet')
    )
    TO 'geracao_temp.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [27]:
import os
os.replace(
    'geracao_temp.parquet',
    'geracao_modificada.parquet'
)

Agora o arquivo modificado contem o ano e o mes.

Cumprindo com o segundo requisito, o arquivo precisa de datas de feriado. 

A biblioteca holidays contem feriados baseados na região, no nosso caso, Brasil 2001-2021.

Com isso, podemos projetar as datas no parquet. Por enquanto, utilizaremos apenas um booleano sobre se é ou não um feriado.

In [28]:
feriados= holidays.Brazil(years= range(2001, 2021))
df_feriados= pd.DataFrame({'data_feriado': pd.to_datetime(list(feriados.keys()))})

In [29]:
con= duckdb.connect()
con.execute("""
    COPY(
        SELECT temp.*, (CAST(temp.din_instante AS DATE) 
            IN (
                SELECT CAST(data_feriado AS DATE) 
                    FROM df_feriados)) 
                    AS feriado
            FROM read_parquet('geracao_modificada.parquet') temp
    )
    TO 'geracao_temp.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

os.replace('geracao_temp.parquet', 'geracao_modificada.parquet')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cumprindo com o terceiro requisito, o arquivo precisa de dados climaticos referente as coordenadas aproximadas perto de grandes usinas. 

O Open-Meteo fornece dados climáticos diários a partir de coordenadas (latitude e longitude), com base na reanálise ERA5.


In [30]:
pontos = {
    'PR': [(-25.41, -54.59), (-26.00, -51.70)],  
    'SC': [(-27.27, -52.38), (-27.60, -51.32)],   
    'RS': [(-28.26, -52.41), (-29.45, -53.28)],   
    'GO': [(-13.83, -48.30), (-18.41, -49.10)],   
    'MT': [(-14.87, -55.79), (-11.86, -55.50)],   
    'MS': [(-20.78, -51.63), (-20.44, -54.65)],   
    'DF': [(-15.79, -47.88)], 
}

In [36]:
URL = "https://archive-api.open-meteo.com/v1/archive"

partes = []

for estado, coords in pontos.items():
    for lat, lon in coords:
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2001-01-01",
            "end_date": "2020-12-31",
            "daily": "precipitation_sum,temperature_2m_mean",
            "timezone": "America/Sao_Paulo",
        }
        r = requests.get(URL, params=params)
        r.raise_for_status()

        d= pd.DataFrame(r.json()['daily'])
        d["id_estado"] = estado
        partes.append(d)
        time.sleep(15)
df_clima = pd.concat(partes, ignore_index=True)
df_clima["time"] = pd.to_datetime(df_clima["time"])
df_clima.to_parquet("clima_diario.parquet")


In [37]:
con.execute("""
    COPY (
        WITH por_estado_dia AS (
            SELECT id_estado, time AS data,
                   AVG(precipitation_sum)   AS chuva_mm,
                   AVG(temperature_2m_mean) AS temp_c
            FROM read_parquet('clima_diario.parquet')
            GROUP BY id_estado, time
        )
        SELECT id_estado,
               EXTRACT(YEAR FROM data)  AS Ano,
               EXTRACT(MONTH FROM data) AS MES,
               SUM(chuva_mm) AS chuva_mm_mes,
               AVG(temp_c)   AS temp_media_mes
        FROM por_estado_dia
        GROUP BY ALL
    )
    TO 'clima_mensal.parquet' (FORMAT PARQUET)
""")


In [38]:
con.execute("""
    COPY (
        SELECT id_estado, Ano, MES,
               SUM(TRY_CAST(val_geracao AS DOUBLE)) AS geracao_total,
               COUNT(DISTINCT CASE WHEN feriado
                     THEN CAST(din_instante AS DATE) END) AS n_feriados
        FROM read_parquet('geracao_modificada.parquet')
        WHERE nom_tipocombustivel = 'Hidráulica'
          AND id_estado IN ('PR','SC','RS','GO','MT','MS','DF')
        GROUP BY id_estado, Ano, MES
    )
    TO 'geracao_mensal.parquet' (FORMAT PARQUET)
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [39]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))           
print(df_final.isna().sum())     


1680
id_estado         0
Ano               0
MES               0
geracao_total     0
n_feriados        0
chuva_mm_mes      0
temp_media_mes    0
dtype: int64


Conforme solicitado pelo professor, o arquivo também recebe as bandeiras tarifárias, obtidas nos dados abertos da ANEEL.

A bandeira é definida mês a mês e vale para o país inteiro, por isso é unida aos dados apenas por ano e mês. Como o sistema de bandeiras só começou em janeiro de 2015, os meses de 2001 a 2014 ficam sem valor.

In [41]:
URL_BANDEIRAS = "https://dadosabertos.aneel.gov.br/dataset/7f43a020-6dc5-44b8-80b4-d97eaa94436c/resource/0591b8f6-fe54-437b-b72b-1aa2efd46e42/download/bandeira-tarifaria-acionamento.csv"

df_bandeiras = pd.read_csv(URL_BANDEIRAS, sep=";", decimal=",")
df_bandeiras["NomBandeiraAcionada"].value_counts()


NomBandeiraAcionada
Verde               69
Vermelha P1         26
Amarela             25
Vermelha P2         15
Escassez Hídrica     8
Name: count, dtype: int64

In [ ]:
df_bandeiras["DatCompetencia"] = pd.to_datetime(df_bandeiras["DatCompetencia"])
df_bandeiras["Ano"] = df_bandeiras["DatCompetencia"].dt.year
df_bandeiras["MES"] = df_bandeiras["DatCompetencia"].dt.month

df_bandeiras = df_bandeiras.rename(columns={
    "NomBandeiraAcionada": "bandeira",
    "VlrAdicionalBandeira": "adicional_bandeira",
})
df_bandeiras = df_bandeiras[["Ano", "MES", "bandeira", "adicional_bandeira"]]
df_bandeiras.to_parquet("bandeiras_mensal.parquet")


In [43]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes,
           b.bandeira, b.adicional_bandeira
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    LEFT JOIN read_parquet('bandeiras_mensal.parquet') b
        USING (Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))
print(df_final.isna().sum())

1680
id_estado                0
Ano                      0
MES                      0
geracao_total            0
n_feriados               0
chuva_mm_mes             0
temp_media_mes           0
bandeira              1176
adicional_bandeira    1176
dtype: int64


Seguindo a recomendação de incluir indicadores econômicos, utilizamos o consumo de energia elétrica do Brasil, total e industrial, em GWh, obtido no Sistema Gerenciador de Séries Temporais (SGS) do Banco Central.

Enquanto o clima representa a água disponível para gerar energia, o consumo representa a demanda. Como a rede elétrica é interligada, os valores são nacionais e também são unidos por ano e mês.


In [44]:
def baixar_sgs(codigo, nome):
    url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
           "?formato=json&dataInicial=01/01/2001&dataFinal=31/12/2020")
    d = pd.read_json(url)
    d["data"] = pd.to_datetime(d["data"], format="%d/%m/%Y")
    d["Ano"] = d["data"].dt.year
    d["MES"] = d["data"].dt.month
    return d.rename(columns={"valor": nome})[["Ano", "MES", nome]]


In [45]:
df_consumo = baixar_sgs(1406, "consumo_total_gwh")
df_industrial = baixar_sgs(1404, "consumo_industrial_gwh")

df_economia = df_consumo.merge(df_industrial, on=["Ano", "MES"])
df_economia.to_parquet("economia_mensal.parquet")
df_economia.head()


,Ano,MES,consumo_total_gwh,consumo_industrial_gwh
0,2001,1,25878,10426
1,2001,2,26410,10856
2,2001,3,26487,10892
3,2001,4,27097,11307
4,2001,5,26311,11338


In [46]:
df_final = con.execute("""
    SELECT g.*, c.chuva_mm_mes, c.temp_media_mes,
           b.bandeira, b.adicional_bandeira,
           e.consumo_total_gwh, e.consumo_industrial_gwh
    FROM read_parquet('geracao_mensal.parquet') g
    LEFT JOIN read_parquet('clima_mensal.parquet') c
        USING (id_estado, Ano, MES)
    LEFT JOIN read_parquet('bandeiras_mensal.parquet') b
        USING (Ano, MES)
    LEFT JOIN read_parquet('economia_mensal.parquet') e
        USING (Ano, MES)
    ORDER BY id_estado, Ano, MES
""").df()

print(len(df_final))
print(df_final.isna().sum())

1680
id_estado                    0
Ano                          0
MES                          0
geracao_total                0
n_feriados                   0
chuva_mm_mes                 0
temp_media_mes               0
bandeira                  1176
adicional_bandeira        1176
consumo_total_gwh            0
consumo_industrial_gwh       0
dtype: int64
